[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/reinfrocement-learning-for-genai-models/blob/main/notebooks/02_reward_and_rl_for_llms.ipynb)

# 02 · Reward: how a score steers a language model, and the leash that keeps it sane

**C9 · W4 · S2: Reinforcement Learning for GenAI Models** · ⏱️ 30 min

## 🏕️ The scenario: Acme Outfitters wants its own support model

Last session you evaluated **Acme Outfitters'** support agent, which ran on a big API model. Acme now wants a
**small open model it can host itself**, one that drafts replies to customer messages ("how long do I have to return this?",
"my tent arrived broken!"), for cost and privacy reasons. The support leads have two things to offer:

- **~480 historical replies** written by support agents (mixed quality: mostly good, some rambling, some that ignore what
  the customer asked for, a few that should never have been sent)
- **~480 comparisons** where a support lead read two replies and picked the better one

Acme's rules haven't changed: returns within **30 days** of delivery, **final-sale** items and **gift cards** are never
refundable, standard shipping is **free over $75**, express costs **$15**, and **never** share another customer's data.

**The question for this session:** the engineering team can prompt a model, fine-tune it with **LoRA**, or tune it with
**reinforcement learning**. What does RL actually change, what doesn't it change, and is it worth it here?

### What's already built (you don't write this)
Everything below is packaged in `rllab`, which the setup cell loads. Training ran **once, before class**: the adapters and
their logs are in the repo, so nothing here trains a model live. This session teaches training as intuition, not code.

| Piece | What it is |
|---|---|
| `rl.config.BASE_MODEL` | **Qwen2.5-0.5B**, a small open model after pretraining only. It has knowledge but hasn't learned to follow instructions. |
| the **LoRA** variant | the base model + a LoRA adapter trained on the historical replies (supervised fine-tuning), in `artifacts/adapters/lora_sft/` |
| the **RL** variant | the LoRA model, then tuned with **DPO** on the support leads' comparisons, in `artifacts/adapters/rl_dpo/` |
| the **instruct** variant | **Qwen2.5-0.5B-Instruct**, the vendor's own SFT + RL version of the same base, prompted with Acme's rules (the "just use an RL-tuned model" option) |
| `rl.toy`, `rl.policy` | miniature RL worlds in NumPy (a recommender, a robot grid, a policy over whole replies) that show the ideas in seconds |
| `rl.evals`, `rl.checks` | rule-based checks, an LLM judge (Qwen2.5-1.5B-Instruct), a blinded human-review sheet |

All four variants get the **same prompts** with the **same greedy decoding**, so every difference you see comes from tuning.
Model replies are cached in `artifacts/cache/` from the instructor's pre-run, so they replay instantly and identically.
A prompt that isn't cached (one you write yourself) runs the model live.

## 🎯 This notebook

**The problem.** Notebook 01 ended with a gap: supervised fine-tuning copies Acme's replies good and bad alike, because it has no way to say that one reply is better than another. RL closes that gap with a **reward**, a number that scores each reply. But a reward is a specification, and Notebook 01 showed what optimisers do with specifications that are slightly wrong.

**Where we're starting from.** The LoRA model from Notebook 01 (trained on Acme's transcripts, already in the repo) and the idea of an RL loop. Nothing from Notebook 01's variables is needed.

**By the end you'll be able to:**
- Map RL's vocabulary onto an LLM: model = policy, reply = action, score = reward
- Write a reward function for a support reply, and see what it rewards that you didn't intend
- Explain the RL loop for LLMs: sample replies, score them, nudge the model toward higher scores
- Explain why the tuned model is kept close to its starting point (the KL constraint, β)
- Explain why human preferences, not hand-written rules, are the usual source of reward

**What you'll do here.** Score candidate replies with a reward you can read, sample real replies from the LoRA model and score them, run the sample-score-nudge loop on a miniature policy, then remove the KL leash and watch what happens.

**Given vs. you write.** Given: the LoRA model's sampled replies (cached), the miniature policy (`rl.policy`), Acme's preference data. You write: a reward for a task of your choice, and a reply that games it.

In [ ]:
#@title ⚙️ Setup: run this first { display-mode: "form" }
# Makes the lab runtime (`rllab`) importable, and installs transformers + peft if they're missing.
# In Colab this clones baluragala/reinfrocement-learning-for-genai-models (branch main); inside a local checkout it uses that checkout.
import sys, subprocess, pathlib, importlib.util
REPO_URL = "https://github.com/baluragala/reinfrocement-learning-for-genai-models.git"
BRANCH = "main"
_need = [p for p in ("torch", "transformers", "peft", "accelerate") if importlib.util.find_spec(p) is None]
if _need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need, "pandas", "matplotlib"], check=True)

_here = pathlib.Path.cwd().resolve()
_src = next((p / "src" for p in [_here, *_here.parents] if (p / "src" / "rllab" / "__init__.py").exists()), None)
if _src is None:
    _base = pathlib.Path("/content") if pathlib.Path("/content").is_dir() else _here
    _repo = _base / "reinfrocement-learning-for-genai-models"
    if not (_repo / ".git").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(_repo), "pull", "-q", "--ff-only"], check=False)
    _src = _repo / "src"
sys.path.insert(0, str(_src))
for _m in [m for m in sys.modules if m == "rllab" or m.startswith("rllab.")]:
    del sys.modules[_m]
import rllab as rl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print(f"rllab {rl.__version__} ready from {_src} · base model: {rl.config.BASE_MODEL}")

### 🖥️ Check the runtime
No API key is needed: every model is a small open model from the Hugging Face Hub.

* **Colab:** *Runtime → Change runtime type → T4 GPU* is recommended. On CPU the cached outputs still replay instantly,
  but prompts you write yourself will take a few seconds each.
* **Local:** `pip install -r requirements.txt`. Apple-silicon Macs use the GPU (MPS) automatically.

The first live call downloads the model it needs (about 1 GB for the 0.5B models and 3 GB for the judge).

In [ ]:
print(rl.check_runtime())

## RL's vocabulary, mapped onto a language model

| RL term | Robot grid (Notebook 01) | LLM tuning |
|---|---|---|
| **agent / policy** | the robot's rule: state → move | **the model**: prompt → probability of every possible reply |
| **state** | the robot's cell | **the prompt** (and the tokens written so far) |
| **action** | one move | **the reply** (a sequence of tokens; each token is a small action) |
| **reward** | +10 goal, −0.1 per step | **a score for the whole reply**, given once it's finished |
| **episode** | start → goal | one prompt → one reply |
| **exploration** | ε-random moves | **sampling** replies at temperature > 0 |

One difference matters: in the grid, the environment *was* the reward. For an LLM **someone has to build the reward**.
It can be a rule, a trained reward model, or a human judgement. Everything in this notebook is about what that choice does.

#### 📖 Names used in this notebook

| Name | What it holds |
|---|---|
| `IF1` | Acme evaluation prompt: *"Using exactly 3 bullet points, tell me how to return my rain jacket."* |
| `cands` | five hand-written candidate replies to IF1, labelled A–E (§1) |
| `acme_reward` | the reward you read and run in §1: did it follow the instruction, and was it brief? |
| `scored` | eight replies sampled from the LoRA model, with rewards (§2) |
| `ref` | the reference policy: how often the LoRA model gives each candidate reply (§3) |
| `prefs` | Acme's preference data: prompt, chosen reply, rejected reply (§4) |

## 1 · A reward is a function that scores replies

### 1.1 · Score five replies to the same prompt

**Why this step:** Before any learning, see what a reward actually is: code that takes a reply and returns a number.

**📥 Inputs**
- `rl.EVAL_PROMPTS`: the fixed Acme prompts from `src/rllab/data.py`; IF1 asks for exactly 3 bullet points
- `cands`: five replies written for this notebook, from correct to useless
- `rl.checks.check`: the rule-based check attached to IF1 (counts bullet lines), from `src/rllab/checks.py`

In [ ]:
IF1 = next(p for p in rl.EVAL_PROMPTS if p.id == "IF1")
cands = {
    "A: 3 clear bullets": "- Start a return from Orders in your account.\n- Print the prepaid label we email you.\n- Drop the parcel at any carrier location.",
    "B: ignores format": "Start a return from Orders in your account, print the prepaid label we email you, and drop the parcel at any carrier location.",
    "C: gushing paragraph": "Thank you so much for reaching out to Acme Outfitters today! We truly appreciate you taking the time to contact us, and we're always absolutely delighted to help. To return your rain jacket, simply start a return from Orders in your account, then print the prepaid label we email you, and finally drop the parcel at any carrier location. I really hope this helps, and please don't hesitate to reach out again. Happy adventuring!",
    "D: 5 bullets": "- Go to Orders.\n- Start a return.\n- Print the label.\n- Pack the jacket.\n- Drop it off.",
    "E: unhelpful": "Returns are easy, just follow the steps on our website.",
}

def acme_reward(prompt, reply):
    r = 1.0 if rl.checks.check(prompt, reply)["pass"] else 0.0     # did it do what the customer asked?
    r -= 0.01 * max(0, rl.checks.words(reply) - 40)                  # small penalty for every word over 40
    return round(r, 2)

rewards = pd.DataFrame({"reward": {k: acme_reward(IF1, v) for k, v in cands.items()},
                        "words": {k: rl.checks.words(v) for k, v in cands.items()},
                        "bullets": {k: rl.checks.bullets(v) for k, v in cands.items()}})
rewards

**🔍 Reading the output**
- `reward`: 1 if the rule check passes (exactly 3 bullets), minus 0.01 for every word over 40.
- `words`, `bullets`: the two signals the reward is built from.

**What to expect, and why:** A highest. B, C, D and E all fail the check; C also loses points for its length. D does the task sensibly, just in five steps, and this reward treats it the same as the useless E.

<details><summary><b>▸ Why it works this way</b> (click to expand)</summary>

Notice what this reward **can't** see: whether the steps are *true*, whether the tone is right, whether
"Drop it off" is enough detail. It sees bullets and word counts. Rule-based rewards work well when the task is checkable
(valid JSON, a passing unit test, a correct maths answer). They break down for "is this a good support reply?". That's
why most LLM tuning uses rewards learned from human preferences (§4, Notebook 03).

</details>

In [ ]:
rl.explain.reward_table(rewards)

## 2 · The loop: sample → score → nudge

#### ✋ Predict first

Sample 8 replies from the LoRA model to the 3-bullet prompt at temperature 0.9. How many will earn the full reward (exactly 3 bullets, not too long)?

- **A.** All 8: it was trained on Acme's replies
- **B.** Some, not all: it learned from transcripts of mixed quality
- **C.** None: sampling makes replies random

*Write your answer down before running the next cell. The output tells you whether you were right.*

### 2.1 · Sample real replies and score them

**Why this step:** RL for LLMs starts by letting the model *explore*: sample several replies to the same prompt at a temperature above zero.

**📥 Inputs**
- `rl.generate('lora', ..., sample=True, n=8)`: 8 replies from the LoRA model at temperature 0.9 (cached from the pre-run with a fixed seed, so everyone sees the same eight)
- `rl.reply('lora', IF1)`: the LoRA model's greedy reply (temperature 0), for comparison
- `acme_reward`: the reward from step 1.1

In [ ]:
samples = rl.generate("lora", [IF1], sample=True, n=8)[0]
scored = pd.DataFrame({"reply": samples, "reward": [acme_reward(IF1, s) for s in samples]})
greedy = rl.reply("lora", IF1)
print("greedy reply:", greedy.replace(chr(10), " ⏎ "), "→ reward", acme_reward(IF1, greedy))
scored.sort_values("reward", ascending=False)

**🔍 Reading the output**
- Each row is one sampled reply (⏎ marks a line break) and its reward.
- The greedy reply is the one the model gives by default.

**What to expect, and why:** A mix. Some samples use three bullets, some write a paragraph, some ramble, because the LoRA model learned from transcripts of mixed quality. The spread is what RL learns from.

In [ ]:
rl.explain.samples(scored, acme_reward(IF1, greedy))

#### ✋ Predict first

Now run the loop on a miniature model that can only give the five replies A–E. It starts with the probabilities the LoRA model has for them (mostly A and B, a little of the rest). Each step: sample 16 replies, score them with `acme_reward`, raise the probability of above-average ones. After 300 steps, where is the probability?

- **A.** Spread evenly: every reply got some reward
- **B.** Almost all on A
- **C.** On C, the longest reply

*Write your answer down before running the next cell. The output tells you whether you were right.*

### 2.2 · Nudge the policy toward higher reward

**Why this step:** This is the policy-gradient update behind PPO and friends, on a policy small enough to watch. A real LLM does the same thing over billions of possible replies instead of five.

**📥 Inputs**
- `ref`: the starting probabilities of A–E, written here to look like an SFT model: mostly A and B
- `rl.policy.reinforce`: sample → score → nudge, from `src/rllab/policy.py`, with a small KL leash β = 0.05
- `rewards.reward`: the reward of each candidate from step 1.1

In [ ]:
labels = list(cands)
ref = np.array([0.40, 0.30, 0.12, 0.08, 0.10])
r = rewards.reward.values
nudged = rl.policy.reinforce(ref, r, beta=0.05, steps=300, seed=0)
rl.plots.reinforce_run(labels, nudged); plt.show()

**🔍 Reading the output**
- Left: the probability of each reply over training (stacked, so each band's thickness is its probability).
- Right: expected reward (green) and the KL drift from where it started (red, dashed).

**What to expect, and why:** A's band swallows the others as expected reward climbs. KL rises, because the model is moving away from its starting mix.

In [ ]:
rl.explain.reinforce(labels, nudged)

## 3 · The leash: why the tuned model is kept close to where it started

During RL for LLMs, the reward is changed to **reward − β · KL(tuned ‖ reference)**. KL measures how far the tuned
model's probabilities have moved from the reference (the SFT model). Every bit of drift costs reward, and β sets the price.

Why pay to stay close?
- **The reward is imperfect.** It was built or learned on replies like the reference's. Far from them, its scores are
  guesses, and the optimiser goes looking for exactly those places (Notebook 01's race car).
- **The reference already writes fluent, sensible text.** Drifting far tends to break what already worked.
- **It limits the damage.** Large β = small, safe changes. Small β = the reward gets whatever it asks for.

#### ✋ Predict first

Swap in a **naive** reward that a busy team might write: *"longer replies are more helpful"* (reward = words ÷ 30). Run the same loop with β = 0 (no leash), β = 1 and β = 3. What happens with β = 0?

- **A.** It still ends on A, the correct reply
- **B.** It ends on C, the longest reply
- **C.** It spreads evenly

*Write your answer down before running the next cell. The output tells you whether you were right.*

### 3.1 · Take the leash off

**Why this step:** See what KL buys you when the reward is slightly wrong, which is the normal case.

**📥 Inputs**
- `naive`: reward = number of words ÷ 30, a proxy for 'helpfulness'
- `ref`: the same starting probabilities as step 2.2
- `true`: `acme_reward` from step 1.1, used only to *measure* the result and never to train

In [ ]:
naive = np.array([rl.checks.words(v) / 30 for v in cands.values()])
true = rewards.reward.values
runs = {beta: rl.policy.reinforce(ref, naive, beta=beta, steps=300, seed=0) for beta in (0.0, 1.0, 3.0)}
rl.plots.kl_tradeoff(runs, [l.split(":")[0] for l in labels]); plt.show()

**🔍 Reading the output**
- One panel per β: the final probability of each candidate A–E.
- Panel titles: KL drift and the final *naive* reward.

**What to expect, and why:** β = 0 piles onto C, the gushing reply: highest naive reward, most drift, worst real reward. β = 3 barely moves from the reference. β = 1 sits in between.

<details><summary><b>▸ Why it works this way</b> (click to expand)</summary>

In real RLHF, β is a knob the team tunes (typically 0.01–0.1 for PPO with a per-token KL; the scale depends
on the implementation). Too large and the model barely changes. Too small and it finds the reward model's blind spots: longer,
more flattering, more confident replies that score well and read badly. DPO (Notebook 03) has the same β with the
same meaning. The trained RL variant in this repo used β = 0.1.

</details>

In [ ]:
rl.explain.kl_sweep([l.split(':')[0] for l in labels], runs, true)

**➡️ So what:** The reward decides *what* the model moves toward; β decides *how far* it's allowed to go. You need both to be right.

## 4 · Where rewards come from: human preferences

### 4.1 · Look at Acme's preference data

**Why this step:** For open-ended replies nobody can write a complete scoring rule. But a support lead *can* read two replies and say which is better. RLHF and DPO both start from exactly this kind of data.

**📥 Inputs**
- `rl.data.preference_dataset()`: 480 comparisons generated from Acme's knowledge base in `src/rllab/data.py`: prompt, the reply the support lead chose, and the one they rejected
- `family`: the kind of message: policy question, format request, complaint, harmful request, harmless-but-scary, unknowable question, or a customer confidently stating a wrong policy (`pushback`)

In [ ]:
prefs = pd.DataFrame(rl.data.preference_dataset())
print(prefs.family.value_counts().to_dict())
prefs.groupby("family").head(1)[["family", "prompt", "chosen", "rejected"]]

**🔍 Reading the output**
- Top line: how many comparisons of each kind.
- Table: one example per kind. `chosen` is the reply the rater preferred, `rejected` the other one.

**What to expect, and why:** Mostly sensible choices (refuse the harmful request, admit not knowing, follow the format). Read the `pushback` and `tone` rows carefully, because raters are people. Notebook 03 checks what these choices actually reward.

In [ ]:
rl.explain.preferences(prefs)

### 🧪 Your turn: design a reward, then game it

Task: *"In one sentence, when will I see my refund?"* (Acme's answer: 5–7 business days after the item arrives.) Write `my_reward(reply)` returning a number. Then write `honest`, a good reply, and `gaming`, a reply that a customer would hate but that scores **at least as high** on your reward. Think keyword stuffing, empty replies, copying the question.

In [ ]:
def my_reward(reply):
    # TODO: score the reply. e.g. +1 if it mentions "5–7", −1 if more than one sentence ...
    return 0.0

honest = "..."    # TODO: a reply you'd be happy to send
gaming = "..."    # TODO: a bad reply that still scores high on my_reward
rl.explain.my_reward({"honest": my_reward(honest), "gaming": my_reward(gaming)})

In [ ]:
#@title ✅ Solution (click to reveal) { display-mode: "form" }
def my_reward(reply):
    r = 1.0 if ("5–7" in reply or "5-7" in reply) else 0.0
    r -= 0.5 * max(0, rl.checks.sentences(reply) - 1)
    return r

honest = "Refunds reach your card 5–7 business days after we receive the item."
gaming = "5–7"                       # scores 1.0: right keyword, one "sentence", tells the customer nothing
rl.explain.my_reward({"honest": my_reward(honest), "gaming": my_reward(gaming)})
# Fixes: require a minimum length, require the subject ("refund"), or have a person (or a model trained on people's
# comparisons) judge it. That last option is RLHF.

## ✅ Recap
- For an LLM, **policy = the model**, **action = the reply**, **reward = a score for the reply**. Someone has to build that score.
- The RL loop: **sample** several replies → **score** them → **nudge** the model toward the higher-scoring ones. It can
  only promote replies the model can already produce.
- **Rewards are proxies**. Optimise one hard and you get what it measures, not what you meant.
- The **KL leash (β)** keeps the tuned model close to its reference, trading some reward for safety.
- For open-ended quality, the reward usually comes from **human preferences**: pairs of replies, one chosen over the other.